Can we fit the CFFs as predicted by a model and generated 

## (1): Import Libraries:

In [92]:
import glob
from sklearn.model_selection import train_test_split
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from iminuit import Minuit
from scipy.integrate import quad
import gepard as g
from gepard.fits import th_KM15
from gepard.bmk import BM10

In [70]:
g.__version__

'0.9.10'

In [97]:
BM10().description

'N/A'

In [98]:
BM10().model

## (2): Generating Pseudodata:

### (2.1): Defining the Kinematic Boundaries:

In [71]:
BEAM_K_LOWER = 5.5
BEAM_K_UPPER = 11.0
Q_SQUARED_LOWER = 1.0
Q_SQUARED_UPPER = 5.0
X_B_LOWER = 0.1
X_B_UPPER = 0.6
T_LOWER = -1.0
T_UPPER = -.1
PHI_LOWER = 0.
PHI_UPPER = 360.

NUMBER_OF_BEAM_K = 6
NUMBER_OF_Q_SQUARED = 10
NUMBER_OF_X_B = 6
NUMBER_OF_T = 6
NUMBER_OF_PHI = 15

K_RANGE = np.linspace(BEAM_K_LOWER, BEAM_K_UPPER, NUMBER_OF_BEAM_K)
Q2_RANGE = np.linspace(Q_SQUARED_LOWER, Q_SQUARED_UPPER, NUMBER_OF_Q_SQUARED)
X_B_RANGE = np.linspace(X_B_LOWER, X_B_UPPER, NUMBER_OF_X_B)
T_RANGE = np.linspace(T_LOWER, T_UPPER, NUMBER_OF_T)
PHI_RANGE = np.linspace(PHI_LOWER, PHI_UPPER, NUMBER_OF_PHI)

In [ ]:
def generate_kinematic_scan():
    rows = []
    set_counter = 0 
    total_settings = len(K_RANGE) * len(Q2_RANGE) * len(X_B_RANGE) * len(T_RANGE) * len(PHI_RANGE)

    print(f"Iterating over {total_settings} total combinations.")

    for fixed_k in K_RANGE:
        for fixed_q_squared in Q2_RANGE:
            for fixed_x_bjorken in X_B_RANGE:
                for fixed_t in T_RANGE:
                    for fixed_phi in PHI_RANGE:
                        set_counter += 1

                        current_datapoint = g.DataPoint(
                                    xB = fixed_x_bjorken, t = fixed_t, Q2 = fixed_q_squared, phi = fixed_phi,
                                    process = "ep2epgamma", exptype = 'fixed target',
                                    in1energy = fixed_k, in1charge = -1, in1polarization = +1,
                                    observable = 'XS',
                                    fname = 'Trento')
                        
                        try:

                            # predict the cffs
                            real_h = th_KM15.ReH(current_datapoint)
                            imag_h = th_KM15.ImH(current_datapoint)
                            real_e = th_KM15.ReH(current_datapoint)
                            imag_e = th_KM15.ImH(current_datapoint)
                            real_ht = th_KM15.ReH(current_datapoint)
                            imag_ht = th_KM15.ImH(current_datapoint)
                            real_et = th_KM15.ReH(current_datapoint)
                            imag_et = th_KM15.ImH(current_datapoint)

                            # predict the cross section:
                            km15_cross_section = th_KM15.predict(current_datapoint)
                            bkm10_cross_section = BM10().XS(current_datapoint)

                            if not (np.isfinite(km15_cross_section) and km15_cross_section >= 0 and not np.isnan(km15_cross_section)):
                                raise ValueError("KM15 cross-section was unphysical.")
                            if not (np.isfinite(bkm10_cross_section) and bkm10_cross_section >= 0 and not np.isnan(bkm10_cross_section)):
                                raise ValueError("BKM10 cross-section was unphysical.")

                        except RuntimeWarning:
                            print("Issue with kinematics detected!")

                        del current_datapoint
                    
    print(f"Has iteration successfully executed: {set_counter + 1 == total_settings}")

### (X): Generate the Pseudodata:
**(WARNING: Running this function requires major computing resources!)**

In [89]:
generate_kinematic_scan()

Iterating over 2160 total combinations.


C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics.py:104: RuntimeWarning: divide by zero encountered in scalar divide
  pt.chi0 = sqrt(2.*pt.Q2)*pt.tK/sqrt(1+pt.eps2)/(pt.Q2+pt.t)
C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics.py:105: RuntimeWarning: divide by zero encountered in scalar divide
  pt.chi = (pt.Q2-pt.t+2.*pt.xB*pt.t)/sqrt(1+pt.eps2)/(pt.Q2+pt.t) - 1.
C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics.py:99: RuntimeWarning: invalid value encountered in sqrt
  pt.K = sqrt(pt.K2)
C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics.py:101: RuntimeWarning: invalid value encountered in sqrt
  pt.tK = sqrt(pt.tK2)
C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics.py:51: RuntimeWarning: invalid value encountered in sqrt
  K = sqrt(K2(Q2, xB, t, y, eps2))
C:\Users\fiore\AppData\Roaming\Python\Python312\site-packages\gepard\kinematics

KeyboardInterrupt: 